# classic_control — DQN on MountainCar / Cartpole / Acrobot

One figure per task: the DQN mean ± 95% bootstrap-CI return over 30 seeds,
each loaded from its own `results/<name>.db` database. Return ranges differ per
task, so each plot fixes its own y-limits.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_HERE = Path.cwd()
_EXP_DIR = _HERE if (_HERE / "config.py").exists() else Path("experiments/classic_control")
sys.path.insert(0, str(_EXP_DIR))
sys.path.insert(0, str(_EXP_DIR.resolve().parents[1]))

from experiment import load_result, load_runs
from analysis.plotting import min_max_normalize, plot_mean_ci, seed_grids_for, style

from config import EXPERIMENT


def grid_for(component, n=500):
    """A shared [0, total_steps] timestep grid from any run's length."""
    df = load_runs(EXPERIMENT, component)
    total_steps = len(load_result(EXPERIMENT, component, df["run_id"][0])["reward"])
    return np.linspace(0, total_steps, n)


In [ ]:
HERE = Path.cwd()
RESULTS = HERE / "results" if (HERE / "results").exists() else Path("experiments/classic_control/results")

# (component name, title, return y-range)
TASKS = [
    ("dqn_mountaincar", "DQN on MountainCar", (-1000, 0)),
    ("dqn_cartpole", "DQN on Cartpole", (0, 500)),
    ("dqn_acrobot", "DQN on Acrobot", (-500, 0)),
]

figures = {}
for name, title, ylim in TASKS:
    grid = grid_for(name)
    stack = seed_grids_for(EXPERIMENT, name, grid)
    n = stack.shape[0]

    fig, ax = plt.subplots(figsize=(9, 6))   # 2:3 height:width
    plot_mean_ci(ax, grid, stack, f"mean over seeds (n={n})", "tab:blue")
    ax.set_title(f"{title}  (mean ± 95% bootstrap CI)")
    ax.legend(loc="lower right", frameon=False)
    style(ax, ylim=ylim)
    fig.tight_layout()
    figures[name] = fig

plt.show()


## Aggregate: normalized return across tasks

Each task's return curves are min-max normalized onto [0, 1], then pooled
across the three tasks; the band is a 95% bootstrap CI over the pooled runs.

In [ ]:
GRID = grid_for(TASKS[0][0])  # every task runs the same number of steps
normalized = [
    min_max_normalize([seed_grids_for(EXPERIMENT, name, GRID)])[0]
    for name, _title, _ylim in TASKS
]

aggregate_fig, ax = plt.subplots(figsize=(9, 6))   # 2:3 height:width
pooled = np.vstack(normalized)
plot_mean_ci(ax, GRID, pooled, f"mean over runs (n={pooled.shape[0]})", "tab:blue")
ax.set_title("DQN on all tasks  (mean ± 95% bootstrap CI)")
ax.legend(loc="lower right", frameon=False)
style(ax, ylim=(0, 1), ylabel="Normalized\nreturn")
aggregate_fig.tight_layout()
plt.show()


In [ ]:
PLOTS_DIR = RESULTS.parent / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"return_{name}.pdf", bbox_inches="tight")
aggregate_fig.savefig(PLOTS_DIR / "return_aggregate.pdf", bbox_inches="tight")
print(f"saved {len(figures) + 1} plot(s) to {PLOTS_DIR}")
